# Train / Dev / Test Splits & Bias–Variance

> Based on Stanford CS230 — C2M1, Andrew Ng / DeepLearning.AI

Choosing the right data splits and diagnosing bias vs variance are the first steps in any practical deep learning project — they tell you *what* to fix before you write a single line of code.

## Learning Objectives

1. Explain why train / dev / test split ratios differ between classic ML and big-data deep learning
2. Diagnose high bias (underfitting) vs high variance (overfitting) from train/dev errors
3. Apply the orthogonalisation principle: fix one problem at a time
4. Understand Bayes optimal error as the lower bound on achievable error

## Data Split Ratios

| Era / Dataset size | Train | Dev | Test |
|---|---|---|---|
| Classic ML (< 10 k) | 60 % | 20 % | 20 % |
| Big data (1 M+) | 98 % | 1 % | 1 % |
| Big data (10 M+) | 99.5 % | 0.25 % | 0.25 % |

> **Rule:** dev and test sets must come from the **same distribution** as data you care about in production. Train set can differ.


## Diagnosing Bias and Variance

Compare **train error** and **dev error** against **Bayes (optimal) error**:

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 680 220" width="680" height="220" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- Column headers -->
  <text x="90"  y="24" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Scenario</text>
  <text x="230" y="24" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Bayes error</text>
  <text x="360" y="24" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Train error</text>
  <text x="490" y="24" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Dev error</text>
  <text x="610" y="24" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Diagnosis</text>
  <line x1="20" y1="30" x2="660" y2="30" stroke="#c8ccd4" stroke-width="1"/>

  <!-- Row 1: High bias -->
  <rect x="20" y="35" width="640" height="40" rx="4" fill="#e05c5c" fill-opacity="0.08"/>
  <text x="90"  y="60" text-anchor="middle" fill="#555" font-size="11">High Bias</text>
  <text x="230" y="60" text-anchor="middle" fill="#555" font-size="11">~1 %</text>
  <text x="360" y="60" text-anchor="middle" fill="#e05c5c" font-size="12" font-weight="bold">15 %</text>
  <text x="490" y="60" text-anchor="middle" fill="#555" font-size="11">16 %</text>
  <text x="610" y="60" text-anchor="middle" fill="#e05c5c" font-size="11">Underfitting — bigger model, train longer</text>

  <!-- Row 2: High variance -->
  <rect x="20" y="80" width="640" height="40" rx="4" fill="#5b9bd5" fill-opacity="0.08"/>
  <text x="90"  y="105" text-anchor="middle" fill="#555" font-size="11">High Variance</text>
  <text x="230" y="105" text-anchor="middle" fill="#555" font-size="11">~1 %</text>
  <text x="360" y="105" text-anchor="middle" fill="#555" font-size="11">1 %</text>
  <text x="490" y="105" text-anchor="middle" fill="#5b9bd5" font-size="12" font-weight="bold">11 %</text>
  <text x="610" y="105" text-anchor="middle" fill="#5b9bd5" font-size="11">Overfitting — regularise, more data</text>

  <!-- Row 3: Both -->
  <rect x="20" y="125" width="640" height="40" rx="4" fill="#f4b942" fill-opacity="0.10"/>
  <text x="90"  y="150" text-anchor="middle" fill="#555" font-size="11">High Bias &amp; Variance</text>
  <text x="230" y="150" text-anchor="middle" fill="#555" font-size="11">~1 %</text>
  <text x="360" y="150" text-anchor="middle" fill="#e05c5c" font-size="12" font-weight="bold">15 %</text>
  <text x="490" y="150" text-anchor="middle" fill="#5b9bd5" font-size="12" font-weight="bold">30 %</text>
  <text x="610" y="150" text-anchor="middle" fill="#f4b942" font-size="11">Both issues</text>

  <!-- Row 4: Just right -->
  <rect x="20" y="170" width="640" height="40" rx="4" fill="#7ecba1" fill-opacity="0.10"/>
  <text x="90"  y="195" text-anchor="middle" fill="#555" font-size="11">Good fit</text>
  <text x="230" y="195" text-anchor="middle" fill="#555" font-size="11">~1 %</text>
  <text x="360" y="195" text-anchor="middle" fill="#7ecba1" font-size="12" font-weight="bold">1.5 %</text>
  <text x="490" y="195" text-anchor="middle" fill="#7ecba1" font-size="12" font-weight="bold">2 %</text>
  <text x="610" y="195" text-anchor="middle" fill="#7ecba1" font-size="11">Low bias, low variance ✓</text>
</svg>

**Avoidable bias** = Train error − Bayes error  
**Variance** = Dev error − Train error


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

np.random.seed(42)
n_train = 30
X_train = np.sort(np.random.uniform(0, 1, n_train))
y_train = np.sin(2 * np.pi * X_train) + np.random.normal(0, 0.2, n_train)

X_test = np.linspace(0, 1, 200)
y_true = np.sin(2 * np.pi * X_test)

degrees = [1, 4, 15]
labels = ['High Bias (degree 1)', 'Good fit (degree 4)', 'High Variance (degree 15)']
colors = [P[1], P[3], P[0]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('Bias–Variance Trade-off: Polynomial Regression', fontsize=13, fontweight='bold')

for ax, deg, label, color in zip(axes, degrees, labels, colors):
    model = make_pipeline(PolynomialFeatures(deg), Ridge(alpha=1e-6))
    model.fit(X_train.reshape(-1, 1), y_train)
    y_pred = model.predict(X_test.reshape(-1, 1))
    y_pred_train = model.predict(X_train.reshape(-1, 1))

    train_err = np.mean((y_pred_train - y_train)**2)
    test_err  = np.mean((y_pred - y_true)**2)

    ax.scatter(X_train, y_train, color='#888', s=25, zorder=3, label='Train data')
    ax.plot(X_test, y_true,  color='#bbb', lw=1.5, ls='--', label='True function')
    ax.plot(X_test, y_pred,  color=color, lw=2.5, label='Model')
    ax.set_title(f'{label}\nTrain MSE={train_err:.3f}  |  Test MSE={test_err:.3f}', fontsize=10)
    ax.set_xlabel('x'); ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=8)
    ax.grid(True)

axes[0].set_ylabel('y')
plt.tight_layout()
plt.show()

# --- Bias-variance decomposition across polynomial degrees ---
degrees_range = range(1, 18)
train_errs, test_errs = [], []
for deg in degrees_range:
    model = make_pipeline(PolynomialFeatures(deg), Ridge(alpha=1e-6))
    model.fit(X_train.reshape(-1, 1), y_train)
    train_errs.append(np.mean((model.predict(X_train.reshape(-1,1)) - y_train)**2))
    test_errs.append(np.mean((model.predict(X_test.reshape(-1,1)) - y_true)**2))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(list(degrees_range), train_errs, color=P[3], lw=2.5, marker='o', ms=5, label='Train error (↓ = high variance risk)')
ax.plot(list(degrees_range), test_errs,  color=P[1], lw=2.5, marker='s', ms=5, label='Test error')
ax.axvline(4, color=P[2], lw=1.5, ls='--', label='Sweet spot (degree 4)')
ax.set_xlabel('Polynomial degree (model complexity)')
ax.set_ylabel('MSE')
ax.set_title('Training vs Test Error across Model Complexity')
ax.legend(fontsize=9)
ax.set_ylim(0, 1.5)
ax.grid(True)
plt.tight_layout()
plt.show()


## Orthogonalisation — Fix One Problem at a Time

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 140" width="600" height="140" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- Step boxes -->
  <rect x="20"  y="40" width="120" height="60" rx="6" fill="#e05c5c" fill-opacity="0.13" stroke="#e05c5c" stroke-width="1.6"/>
  <rect x="180" y="40" width="120" height="60" rx="6" fill="#5b9bd5" fill-opacity="0.13" stroke="#5b9bd5" stroke-width="1.6"/>
  <rect x="340" y="40" width="120" height="60" rx="6" fill="#7ecba1" fill-opacity="0.13" stroke="#7ecba1" stroke-width="1.6"/>
  <rect x="500" y="40" width="80"  height="60" rx="6" fill="#f4b942" fill-opacity="0.13" stroke="#f4b942" stroke-width="1.6"/>
  <!-- Labels -->
  <text x="80"  y="64"  text-anchor="middle" fill="#b03a3a" font-size="11" font-weight="bold">High Bias?</text>
  <text x="80"  y="80"  text-anchor="middle" fill="#b03a3a" font-size="10">Bigger model</text>
  <text x="80"  y="93"  text-anchor="middle" fill="#b03a3a" font-size="10">Train longer</text>
  <text x="240" y="64"  text-anchor="middle" fill="#3a7bbf" font-size="11" font-weight="bold">High Variance?</text>
  <text x="240" y="80"  text-anchor="middle" fill="#3a7bbf" font-size="10">More data</text>
  <text x="240" y="93"  text-anchor="middle" fill="#3a7bbf" font-size="10">Regularisation</text>
  <text x="400" y="64"  text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Dev ≈ Test?</text>
  <text x="400" y="80"  text-anchor="middle" fill="#3d7a5e" font-size="10">Check distribution</text>
  <text x="400" y="93"  text-anchor="middle" fill="#3d7a5e" font-size="10">mismatch</text>
  <text x="540" y="74"  text-anchor="middle" fill="#a07010" font-size="11" font-weight="bold">Deploy ✓</text>
  <!-- Arrows -->
  <line x1="140" y1="70" x2="177" y2="70" stroke="#888" stroke-width="1.5"/>
  <polygon points="178,70 170,66 170,74" fill="#888"/>
  <line x1="300" y1="70" x2="337" y2="70" stroke="#888" stroke-width="1.5"/>
  <polygon points="338,70 330,66 330,74" fill="#888"/>
  <line x1="460" y1="70" x2="497" y2="70" stroke="#888" stroke-width="1.5"/>
  <polygon points="498,70 490,66 490,74" fill="#888"/>
  <!-- Step labels -->
  <text x="80"  y="118" text-anchor="middle" fill="#888" font-size="10">Step 1</text>
  <text x="240" y="118" text-anchor="middle" fill="#888" font-size="10">Step 2</text>
  <text x="400" y="118" text-anchor="middle" fill="#888" font-size="10">Step 3</text>
  <text x="540" y="118" text-anchor="middle" fill="#888" font-size="10">Step 4</text>
</svg>

Always resolve **bias first**, then variance. Trying to reduce variance before eliminating bias wastes effort.
